In [1]:
from src.data_loading import load_olist, validate_tables
from src.features import build_customer_features, validate_features, preprocessing_diagnostics
from src.clustering import run_clustering

# Load the Olist tables
tables = load_olist()

# Validate the tables
validate_tables(tables) # set verbose=False to disable printing of validation results


────────────────────────────────────────────────────────────────────────────────────────
Data Quality Report
────────────────────────────────────────────────────────────────────────────────────────

Informational
  [INFO] order_reviews: 1,603 rows where a review_id is linked to more than one order_id. This is expected (the same review can apply to multiple orders); no action needed.

Needs attention
  [WARN] order_reviews: 1,098 rows where an order_id has multiple distinct reviews (revised/follow-up reviews). Feature engineering must pick one review per order before customer-level aggregation (recommended: latest review_answer_timestamp).
  [WARN] products: 610 rows with missing product_category_name values.
  [WARN] orders: order_approved_at contains 160 missing values.
  [WARN] orders: order_delivered_carrier_date contains 1,783 missing values.
  [WARN] orders: order_delivered_customer_date contains 2,965 missing values.

──────────────────────────────────────────────────────────────

In [2]:
# Build the customer features
feats = build_customer_features(tables)

# Validate the features and compute preprocessing diagnostics
validate_features(feats, feats.attrs["category_columns"])
preprocessing_diagnostics(feats)


Preprocessing diagnostics
[latent_space]
n_customers_used                     92,747
n_customers_excluded_missing_extras  611
block_sigma1:
  rfm       328.4736
  category  321.1816
  extras    343.0620
n_raw_features                       31
d90                                  26
d_max                                26
d_star                               26
var_target                           0.9000
realized_variance                    0.9026
cap_binding                          False

[dbscan_eps]
eps          0.0155
min_samples  52

[monetary_skew]
raw_skew                   9.2126
log1p_skew                 0.5313
skew_reduction             8.6812
monetary_raw_max_over_p75  74.8676

[category_selection]
target_coverage     0.9000
n_categories        25
achieved_coverage   0.9082
n_categories_total  71



In [3]:
# Run Clustering
result = run_clustering(feats)

# Validate the clustering results
print("Clustering results:")
result["kmeans"]["labels"]       # chosen-K labels, aligned to result["customer_ids"]
result["kmeans"]["k_sweep"]      # DataFrame: k, inertia, silhouette -- for the elbow plot
result["dbscan"]["labels"]       # -1 = noise
